In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
dbutils.widgets.text("inc_flag","0")

In [0]:
inc_flag = dbutils.widgets.get("inc_flag")
print(inc_flag)


In [0]:
df_src = spark.sql('''
select distinct(Model_ID) as Model_ID, model_category from parquet.`abfss://silver@storageaccountjan2026.dfs.core.windows.net/carsales`
''')
df_src.display()

In [0]:
if spark.catalog.tableExists('cars_catalog.gols.dim_model') :
    df_sink = spark.sql('''
                        select dim_model_key,Model_ID, model_category 
                        from parquet.`abfss://silver@storageaccountjan2026.dfs.core.windows.net/carsales' 
                        ''')

else :
    df_sink = spark.sql('''select 1 as dim_model_key,Model_ID, model_category from parquet.`abfss://silver@storageaccountjan2026.dfs.core.windows.net/carsales` where 1=0 ''')




In [0]:
df_sink.display()

In [0]:
df_filter = df_src.join(df_sink,df_src.Model_ID == df_sink.Model_ID,'left').select(df_src.Model_ID,df_src.model_category,df_sink.dim_model_key)
df_filter.display()

In [0]:
df_filter_old = df_filter.filter(df_filter.dim_model_key.isNotNull())
df_filter_old.display()


In [0]:
df_filter_new = df_filter.filter(df_filter.dim_model_key.isNull()).select("Model_ID","model_category")
df_filter_new.display()

In [0]:
if (inc_flag == "0") :
    max_value = 1
else :
    max_value_df = spark.sql('select max(dim_model_key) from cars_catalog.gold.dim_model')
    max_value = max_value_df.collect()[0][0]


In [0]:
df_filter_new = df_filter_new.withColumn("dim_model_key",max_value+monotonically_increasing_id())
df_filter_new.display()

In [0]:
df_final = df_filter_old.union(df_filter_new)

In [0]:
from delta.tables import DeltaTable


In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_model'):
    delta_tbl = DeltaTable.forPath(spark,'abfss://gold@storageaccountjan2026.dfs.core.windows.net/dim_model')
    delta_tbl.alias("trg").merge(df_filter_new.alias("src"),"trg.dim_model_key = src.dim_model_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .excute()

else :
    df_final.write.format("delta")\
        .mode("overwrite")\
            .option("path","abfss://gold@storageaccountjan2026.dfs.core.windows.net/dim_model")\
                .saveAsTable("cars_catalog.gold.dim_model")

In [0]:
%sql select * from cars_catalog.gold.dim_model